# 🏥 CNN-Based ICD-10 Auto-Coding System
## Notebook 2: Text Preprocessing & Data Preparation

---

**This Notebook Covers**:
1. Load extracted data from Notebook 1
2. Text cleaning and normalization
3. Tokenization and vocabulary building
4. Label encoding for ICD-10 codes
5. Train/Validation/Test split
6. Save preprocessed data for training

**Input**: `all_documents_extracted.csv` (1,063 docs)
**Output**: Tokenized sequences, label matrices, vocabulary

**Estimated Runtime**: ~30-45 minutes (CPU)

---

## 📋 Section 1: Setup & Load Data

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted!")

In [ ]:
%%capture
# Install required packages
!pip install nltk gensim
print("✓ Packages installed!")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import json
import re
import os
import pickle
from collections import Counter
from typing import List, Dict, Tuple, Set
from tqdm import tqdm

# NLP
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

print("✓ All imports successful!")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive"
PROJECT_FOLDER = f"{DRIVE_BASE}/ICD10_Project"
DATA_FOLDER = f"{PROJECT_FOLDER}/data"
PROCESSED_FOLDER = f"{DATA_FOLDER}/processed"

# Input file (from Notebook 1)
INPUT_CSV = f"{PROCESSED_FOLDER}/all_documents_extracted.csv"

# Output paths
TRAIN_TEST_FOLDER = f"{DATA_FOLDER}/train_test_split"
VOCAB_FILE = f"{TRAIN_TEST_FOLDER}/vocabulary.pkl"
LABEL_ENCODER_FILE = f"{TRAIN_TEST_FOLDER}/label_encoder.pkl"
PREPROCESSED_DATA_FILE = f"{TRAIN_TEST_FOLDER}/preprocessed_data.pkl"

# Preprocessing settings
MAX_SEQUENCE_LENGTH = 2000  # Max tokens per document
MIN_TOKEN_FREQ = 5          # Minimum frequency to include in vocab
TOP_N_CODES = 100           # Number of ICD-10 codes to predict
MIN_CODE_FREQ = 10          # Minimum occurrences for a code

# Train/Val/Test split
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

# Create output directory
os.makedirs(TRAIN_TEST_FOLDER, exist_ok=True)

print("Configuration set!")
print(f"  Input: {INPUT_CSV}")
print(f"  Output: {TRAIN_TEST_FOLDER}")
print(f"  Max sequence length: {MAX_SEQUENCE_LENGTH}")
print(f"  Top N codes: {TOP_N_CODES}")

In [ ]:
# Load extracted data
print("Loading extracted data...")
df = pd.read_csv(INPUT_CSV)

print(f"\nDataset loaded!")
print(f"  Documents: {len(df)}")
print(f"  Columns: {list(df.columns)}")

# Parse ICD-10 codes from JSON strings
def parse_codes(codes_str):
    try:
        if isinstance(codes_str, str):
            return json.loads(codes_str)
        return codes_str if codes_str else []
    except:
        return []

df['icd10_codes_list'] = df['icd10_codes'].apply(parse_codes)
df['num_codes'] = df['icd10_codes_list'].apply(len)

# Basic stats
print(f"\nCode Statistics:")
print(f"  Total codes: {df['num_codes'].sum()}")
print(f"  Avg codes/doc: {df['num_codes'].mean():.1f}")
print(f"  Docs with codes: {(df['num_codes'] > 0).sum()}")

---

## 🔧 Section 2: Text Preprocessing

In [ ]:
class MedicalTextPreprocessor:
    """
    Preprocesses medical text for CNN input.
    - Cleans and normalizes text
    - Removes PHI patterns
    - Tokenizes and lemmatizes
    - Preserves medical terminology
    """
    
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        
        # Medical terms to preserve (don't remove as stopwords)
        self.medical_keep = {
            'pain', 'left', 'right', 'upper', 'lower', 'chronic', 'acute',
            'mild', 'moderate', 'severe', 'normal', 'abnormal', 'positive',
            'negative', 'stable', 'unstable', 'primary', 'secondary'
        }
        self.stop_words -= self.medical_keep
        
        # PHI patterns to remove
        self.phi_patterns = [
            r'\b\d{3}-\d{2}-\d{4}\b',  # SSN
            r'\b\d{10}\b',              # Phone
            r'\b[A-Z][a-z]+\s+[A-Z][a-z]+\b',  # Names (basic)
            r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',   # Dates
        ]
    
    def clean_text(self, text: str) -> str:
        """Clean and normalize text"""
        if not text or not isinstance(text, str):
            return ""
        
        # Lowercase
        text = text.lower()
        
        # Remove PHI patterns
        for pattern in self.phi_patterns:
            text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
        
        # Normalize medical abbreviations
        text = re.sub(r'\bpt\b', 'patient', text)
        text = re.sub(r'\bhx\b', 'history', text)
        text = re.sub(r'\bdx\b', 'diagnosis', text)
        text = re.sub(r'\brx\b', 'prescription', text)
        text = re.sub(r'\btx\b', 'treatment', text)
        text = re.sub(r'\bsx\b', 'symptoms', text)
        
        # Remove special characters but keep hyphens in medical terms
        text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
        
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def tokenize(self, text: str) -> List[str]:
        """Tokenize and lemmatize text"""
        if not text:
            return []
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Filter and lemmatize
        processed = []
        for token in tokens:
            # Skip short tokens and stopwords
            if len(token) < 2:
                continue
            if token in self.stop_words:
                continue
            if token.isdigit() and len(token) > 4:  # Skip long numbers
                continue
            
            # Lemmatize
            lemma = self.lemmatizer.lemmatize(token)
            processed.append(lemma)
        
        return processed
    
    def preprocess(self, text: str) -> List[str]:
        """Full preprocessing pipeline"""
        cleaned = self.clean_text(text)
        tokens = self.tokenize(cleaned)
        return tokens


# Initialize preprocessor
preprocessor = MedicalTextPreprocessor()

# Test
test_text = "Patient John Doe DOB 01/15/1950 presents with chronic lower back pain and Type 2 DM."
tokens = preprocessor.preprocess(test_text)
print(f"Test preprocessing:")
print(f"  Input: {test_text[:60]}...")
print(f"  Tokens: {tokens}")
print("\n✓ MedicalTextPreprocessor ready!")

In [ ]:
# Preprocess all documents
print("Preprocessing all documents...")
print("This may take 10-15 minutes.\n")

tokenized_docs = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Tokenizing"):
    text = row['full_text']
    tokens = preprocessor.preprocess(text)
    tokenized_docs.append(tokens)

df['tokens'] = tokenized_docs
df['token_count'] = df['tokens'].apply(len)

print(f"\nPreprocessing complete!")
print(f"  Documents processed: {len(df)}")
print(f"  Avg tokens/doc: {df['token_count'].mean():.0f}")
print(f"  Min tokens: {df['token_count'].min()}")
print(f"  Max tokens: {df['token_count'].max()}")

---

## 📚 Section 3: Build Vocabulary

In [ ]:
class Vocabulary:
    """
    Vocabulary for converting tokens to indices.
    """
    
    def __init__(self, min_freq: int = 5):
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.word_freq = Counter()
    
    def build(self, tokenized_docs: List[List[str]]):
        """Build vocabulary from tokenized documents"""
        # Count all words
        for tokens in tokenized_docs:
            self.word_freq.update(tokens)
        
        # Add frequent words to vocab
        idx = 2  # Start after PAD and UNK
        for word, freq in self.word_freq.most_common():
            if freq >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
        
        return self
    
    def encode(self, tokens: List[str], max_length: int = None) -> List[int]:
        """Convert tokens to indices"""
        indices = [self.word2idx.get(t, 1) for t in tokens]  # 1 = UNK
        
        if max_length:
            if len(indices) > max_length:
                indices = indices[:max_length]
            else:
                indices = indices + [0] * (max_length - len(indices))  # Pad
        
        return indices
    
    def decode(self, indices: List[int]) -> List[str]:
        """Convert indices back to tokens"""
        return [self.idx2word.get(i, '<UNK>') for i in indices if i != 0]
    
    def __len__(self):
        return len(self.word2idx)


# Build vocabulary
print("Building vocabulary...")
vocab = Vocabulary(min_freq=MIN_TOKEN_FREQ)
vocab.build(df['tokens'].tolist())

print(f"\nVocabulary Statistics:")
print(f"  Total unique tokens: {len(vocab.word_freq)}")
print(f"  Vocabulary size (freq >= {MIN_TOKEN_FREQ}): {len(vocab)}")
print(f"\nTop 20 most frequent words:")
for word, freq in vocab.word_freq.most_common(20):
    print(f"  {word}: {freq}")

In [ ]:
# Encode all documents
print(f"Encoding documents (max_length={MAX_SEQUENCE_LENGTH})...")

encoded_docs = []
for tokens in tqdm(df['tokens'], desc="Encoding"):
    encoded = vocab.encode(tokens, max_length=MAX_SEQUENCE_LENGTH)
    encoded_docs.append(encoded)

X = np.array(encoded_docs, dtype=np.int32)

print(f"\nEncoded shape: {X.shape}")
print(f"  Documents: {X.shape[0]}")
print(f"  Sequence length: {X.shape[1]}")
print(f"  Memory: {X.nbytes / 1024 / 1024:.1f} MB")

---

## 🏷️ Section 4: Label Encoding

In [ ]:
# Analyze code frequencies
all_codes = []
for codes in df['icd10_codes_list']:
    all_codes.extend(codes)

code_freq = Counter(all_codes)

print("ICD-10 Code Frequency Analysis")
print("=" * 50)
print(f"Total code occurrences: {len(all_codes)}")
print(f"Unique codes: {len(code_freq)}")

# Filter by minimum frequency
frequent_codes = {code for code, freq in code_freq.items() if freq >= MIN_CODE_FREQ}
print(f"\nCodes with >= {MIN_CODE_FREQ} occurrences: {len(frequent_codes)}")

# Get top N codes
top_codes = [code for code, freq in code_freq.most_common(TOP_N_CODES)]
print(f"Top {TOP_N_CODES} codes selected for training")

print(f"\nTop 10 codes:")
for i, (code, freq) in enumerate(code_freq.most_common(10), 1):
    pct = freq / len(df) * 100
    print(f"  {i}. {code}: {freq} ({pct:.1f}%)")

In [ ]:
# Create multi-label encoder
print(f"Creating label encoder for top {TOP_N_CODES} codes...")

# Filter codes to only include top N
top_codes_set = set(top_codes)

def filter_codes(codes_list):
    """Keep only top N codes"""
    return [c for c in codes_list if c in top_codes_set]

df['filtered_codes'] = df['icd10_codes_list'].apply(filter_codes)
df['num_filtered'] = df['filtered_codes'].apply(len)

# Multi-label binarizer
mlb = MultiLabelBinarizer(classes=top_codes)
Y = mlb.fit_transform(df['filtered_codes'])

print(f"\nLabel matrix shape: {Y.shape}")
print(f"  Documents: {Y.shape[0]}")
print(f"  Classes (codes): {Y.shape[1]}")
print(f"\nLabel Statistics:")
print(f"  Avg labels/doc: {Y.sum(axis=1).mean():.1f}")
print(f"  Docs with 0 labels: {(Y.sum(axis=1) == 0).sum()}")
print(f"  Label sparsity: {(Y == 0).sum() / Y.size * 100:.1f}%")

In [ ]:
# Create code to index mapping
code2idx = {code: idx for idx, code in enumerate(mlb.classes_)}
idx2code = {idx: code for code, idx in code2idx.items()}

# Store label encoder info
label_encoder = {
    'mlb': mlb,
    'code2idx': code2idx,
    'idx2code': idx2code,
    'classes': list(mlb.classes_),
    'n_classes': len(mlb.classes_),
    'code_freq': {code: code_freq[code] for code in mlb.classes_}
}

print(f"Label encoder created with {label_encoder['n_classes']} classes")

---

## ✂️ Section 5: Train/Val/Test Split

In [ ]:
# Split data
print(f"Splitting data: {TRAIN_RATIO*100:.0f}% train, {VAL_RATIO*100:.0f}% val, {TEST_RATIO*100:.0f}% test")

# First split: train vs (val + test)
X_train, X_temp, Y_train, Y_temp, idx_train, idx_temp = train_test_split(
    X, Y, df.index.values,
    test_size=(VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=None  # Multi-label, can't stratify easily
)

# Second split: val vs test
val_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
X_val, X_test, Y_val, Y_test, idx_val, idx_test = train_test_split(
    X_temp, Y_temp, idx_temp,
    test_size=val_test_ratio,
    random_state=RANDOM_SEED
)

print(f"\nSplit Results:")
print(f"  Train: {len(X_train)} documents ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val:   {len(X_val)} documents ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test:  {len(X_test)} documents ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nLabel Distribution:")
print(f"  Train avg labels/doc: {Y_train.sum(axis=1).mean():.1f}")
print(f"  Val avg labels/doc: {Y_val.sum(axis=1).mean():.1f}")
print(f"  Test avg labels/doc: {Y_test.sum(axis=1).mean():.1f}")

In [ ]:
# Verify label coverage
train_labels = set(np.where(Y_train.sum(axis=0) > 0)[0])
val_labels = set(np.where(Y_val.sum(axis=0) > 0)[0])
test_labels = set(np.where(Y_test.sum(axis=0) > 0)[0])

print("Label Coverage Check:")
print(f"  Labels in train: {len(train_labels)}/{TOP_N_CODES}")
print(f"  Labels in val: {len(val_labels)}/{TOP_N_CODES}")
print(f"  Labels in test: {len(test_labels)}/{TOP_N_CODES}")

# Check for labels not in train
missing_in_train = set(range(TOP_N_CODES)) - train_labels
if missing_in_train:
    print(f"\n⚠️ Warning: {len(missing_in_train)} labels not in training set!")
    for idx in list(missing_in_train)[:5]:
        print(f"    {idx2code[idx]}")
else:
    print("\n✓ All labels present in training set!")

---

## 💾 Section 6: Save Preprocessed Data

In [ ]:
# Save vocabulary
print("Saving vocabulary...")
with open(VOCAB_FILE, 'wb') as f:
    pickle.dump(vocab, f)
print(f"  ✓ Saved to {VOCAB_FILE}")

# Save label encoder
print("\nSaving label encoder...")
with open(LABEL_ENCODER_FILE, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"  ✓ Saved to {LABEL_ENCODER_FILE}")

# Save preprocessed data
print("\nSaving preprocessed data...")
preprocessed_data = {
    'X_train': X_train,
    'X_val': X_val,
    'X_test': X_test,
    'Y_train': Y_train,
    'Y_val': Y_val,
    'Y_test': Y_test,
    'idx_train': idx_train,
    'idx_val': idx_val,
    'idx_test': idx_test,
    'vocab_size': len(vocab),
    'n_classes': Y.shape[1],
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'top_n_codes': TOP_N_CODES
}

with open(PREPROCESSED_DATA_FILE, 'wb') as f:
    pickle.dump(preprocessed_data, f)
print(f"  ✓ Saved to {PREPROCESSED_DATA_FILE}")

In [ ]:
# Save summary statistics
summary = {
    'preprocessing_date': pd.Timestamp.now().isoformat(),
    'total_documents': len(df),
    'vocab_size': len(vocab),
    'max_sequence_length': MAX_SEQUENCE_LENGTH,
    'n_classes': TOP_N_CODES,
    'min_code_freq': MIN_CODE_FREQ,
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'avg_tokens_per_doc': float(df['token_count'].mean()),
    'avg_labels_per_doc': float(Y.sum(axis=1).mean()),
    'label_sparsity': float((Y == 0).sum() / Y.size),
    'top_10_codes': [{'code': code, 'freq': freq} 
                    for code, freq in code_freq.most_common(10)]
}

summary_file = f"{TRAIN_TEST_FOLDER}/preprocessing_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Summary saved to {summary_file}")

---

## ✅ Section 7: Final Summary

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 2 COMPLETE: TEXT PREPROCESSING")
print("="*60)

print("\n📊 Dataset Statistics:")
print(f"  Documents: {len(df)}")
print(f"  Vocabulary size: {len(vocab):,}")
print(f"  Max sequence length: {MAX_SEQUENCE_LENGTH}")
print(f"  Number of classes: {TOP_N_CODES}")

print("\n📁 Saved Files:")
print(f"  {VOCAB_FILE}")
print(f"  {LABEL_ENCODER_FILE}")
print(f"  {PREPROCESSED_DATA_FILE}")
print(f"  {summary_file}")

print("\n📈 Split Summary:")
print(f"  Train: {len(X_train)} docs")
print(f"  Val: {len(X_val)} docs")
print(f"  Test: {len(X_test)} docs")

print("\n📌 Next Steps:")
print("  1. Run Notebook 3: CNN Model Training")
print("  2. Use GPU runtime for faster training!")
print("  3. Expected training time: 30-60 min with T4 GPU")

print("\n" + "="*60)